# US Superstore Business Intelligence Report

Comprehensive analysis of the Superstore dataset using Pandas, Matplotlib, Seaborn and ipywidgets.

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from ipywidgets import interact, Dropdown, IntSlider
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('data/Sample - Superstore.csv', encoding='latin-1')
print('Shape:', df.shape)
df.head()


## 1. Data Exploration & Cleaning

In [ ]:

df.info()
df.isnull().sum()


In [ ]:

print('Duplicates:', df.duplicated().sum())
df = df.drop_duplicates()

if 'Postal Code' in df.columns:
    df['Postal Code'] = df['Postal Code'].fillna(0)

df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

df['Profit Margin'] = (df['Profit'] / df['Sales']) * 100
df['Order Year'] = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.month
df['Order Month-Year'] = df['Order Date'].dt.to_period('M')

df.head()


## 2. Monthly Sales Trend

In [ ]:

monthly_sales = df.groupby(['Order Month-Year','Category'])['Sales'].sum().reset_index()
monthly_sales['Date'] = monthly_sales['Order Month-Year'].dt.to_timestamp()

def plot_monthly_sales(category='All'):
    plt.figure(figsize=(12,6))

    if category == 'All':
        total_monthly = df.groupby('Order Month-Year')['Sales'].sum()
        plt.plot(total_monthly.index.to_timestamp(), total_monthly.values, marker='o')
    else:
        data = monthly_sales[monthly_sales['Category'] == category]
        plt.plot(data['Date'], data['Sales'], marker='o')

    plt.title(f'Monthly Sales Trend - {category}')
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.show()

categories = ['All'] + list(df['Category'].unique())
interact(plot_monthly_sales, category=Dropdown(options=categories));


## 3. Geographic Analysis

In [ ]:

state_sales = df.groupby('State')['Sales'].sum().sort_values()

def plot_top_states(top_n=10):
    top_states = state_sales.tail(top_n)

    plt.figure(figsize=(12,6))
    plt.barh(top_states.index, top_states.values)
    plt.title(f'Top {top_n} States by Sales')
    plt.show()

interact(plot_top_states, top_n=IntSlider(min=5,max=25,value=10));


## 4. Top 10 Profitable Products

In [ ]:

product_profit = df.groupby('Product Name')['Profit'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(12,6))
ax = sns.barplot(x=product_profit.values, y=product_profit.index)

for i,v in enumerate(product_profit.values):
    ax.text(v, i, f'${v:,.0f}')

plt.title('Top 10 Most Profitable Products')
plt.show()


## 5. Discount vs Profit

In [ ]:

plt.figure(figsize=(12,7))

sns.scatterplot(data=df, x='Discount', y='Profit', hue='Category')
sns.regplot(data=df, x='Discount', y='Profit', scatter=False, color='red')

plt.axhline(0, linestyle='--')
plt.title('Discount vs Profit')
plt.show()


## 6. Executive Summary

In [ ]:

total_sales = df['Sales'].sum()
total_profit = df['Profit'].sum()

print('Total Sales:', round(total_sales,2))
print('Total Profit:', round(total_profit,2))
print('Profit Margin (%):', round((total_profit/total_sales)*100,2))

top_state = df.groupby('State')['Sales'].sum().idxmax()
print('Top State:', top_state)

top_category = df.groupby('Category')['Sales'].sum().idxmax()
print('Top Category:', top_category)
